# Architecture comparison — run5 (hidden_dim / num_conv_blocks) vs run4a baseline

Compares 5 runs on one set of clean, headline charts (one line per run - not overloaded)
plus a per-vocab_size convergence grid (split out for readability, since 5 architectures x
15 configs would be unreadable in one chart).

All 5 runs share the same tokenizer grid, `--max-token-len 9`, `--pad-layout anchored`,
individual per-vocab_size `max_len` - only the architecture (`hidden_dim`/`num_conv_blocks`)
differs between them.

In [ ]:
# parameters
RUNS = {
    "baseline (h192 d4)": "../artifacts/compare_forward_tokenizers_2026-08-01_fine_grid_individual_maxlen_maxtokenlen9_anchored_40epochs",
    "h128 d4":  "../artifacts/compare_forward_tokenizers_2026-08-01_fine_grid_individual_maxlen_maxtokenlen9_anchored_h128_d4_40epochs",
    "h192 d3":  "../artifacts/compare_forward_tokenizers_2026-08-01_fine_grid_individual_maxlen_maxtokenlen9_anchored_h192_d3_40epochs",
    "h128 d3":  "../artifacts/compare_forward_tokenizers_2026-08-01_fine_grid_individual_maxlen_maxtokenlen9_anchored_h128_d3_40epochs",
    "h96 d3":   "../artifacts/compare_forward_tokenizers_2026-08-01_fine_grid_individual_maxlen_maxtokenlen9_anchored_h96_d3_40epochs",
}
RUN_COLORS = {
    "baseline (h192 d4)": "black",
    "h128 d4": "tab:blue",
    "h192 d3": "tab:orange",
    "h128 d3": "tab:green",
    "h96 d3": "tab:red",
}

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DEFAULT_HIDDEN_DIM = 192
DEFAULT_NUM_CONV_BLOCKS = 4


def load_run(run_label, output_root):
    results_dir = Path(output_root)
    raw = pd.read_csv(results_dir / "raw_results.csv")
    for col, default in [("max_token_len", None), ("pad_layout", "end"),
                          ("hidden_dim", DEFAULT_HIDDEN_DIM), ("num_conv_blocks", DEFAULT_NUM_CONV_BLOCKS)]:
        if col not in raw.columns:
            raw[col] = default
    raw["run_label"] = run_label
    with open(results_dir / "summary.json") as f:
        summary = json.load(f)
    for row in summary:
        row["run_label"] = run_label
    return raw, summary, results_dir


all_raw, all_summary, run_dirs = [], [], {}
for run_label, output_root in RUNS.items():
    raw, summary, results_dir = load_run(run_label, output_root)
    all_raw.append(raw)
    all_summary.extend(summary)
    run_dirs[run_label] = results_dir

raw = pd.concat(all_raw, ignore_index=True)
print(f"Loaded {len(raw)} runs total across {len(RUNS)} architecture variants")
raw.groupby("run_label")["vocab_size"].count().rename("rows")

## Summary table: main metrics per (run, tokenizer, vocab_size)

In [ ]:
def fmt(mean_std, precision=4):
    mean, std = mean_std
    return f"{mean:.{precision}f} \u00b1 {std:.{precision}f}"

summary_df = pd.DataFrame([
    {
        "run": row["run_label"],
        "tokenizer": row["tokenizer"],
        "vocab_size": row["vocab_size"],
        "test_loss": fmt(row["test_loss"]),
        "test_cosine": fmt(row["test_cosine"]),
        "best_val_loss": fmt(row["best_val_loss"]),
        "epochs_to_best_val": fmt(row["epochs_to_best_val"], precision=1),
        "num_parameters": row["num_parameters"],
        "n_seeds": row["n_seeds"],
    }
    for row in all_summary
])
summary_df.sort_values(["tokenizer", "vocab_size", "run"])

## test_loss vs vocab_size, one line per architecture

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for run_label in RUNS:
    wp = sorted((r for r in all_summary if r["run_label"] == run_label and r["tokenizer"] == "wordpiece"),
                key=lambda r: r["vocab_size"])
    vocab_sizes = [r["vocab_size"] for r in wp]
    means = [r["test_loss"][0] for r in wp]
    stds = [r["test_loss"][1] for r in wp]
    ax.errorbar(vocab_sizes, means, yerr=stds, marker="o", capsize=3, label=run_label, color=RUN_COLORS[run_label])

ax.set_xlabel("vocab_size")
ax.set_ylabel("test_loss")
ax.set_title("test_loss vs vocab_size, by architecture")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## test_cosine vs vocab_size, one line per architecture

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for run_label in RUNS:
    wp = sorted((r for r in all_summary if r["run_label"] == run_label and r["tokenizer"] == "wordpiece"),
                key=lambda r: r["vocab_size"])
    vocab_sizes = [r["vocab_size"] for r in wp]
    means = [r["test_cosine"][0] for r in wp]
    stds = [r["test_cosine"][1] for r in wp]
    ax.errorbar(vocab_sizes, means, yerr=stds, marker="o", capsize=3, label=run_label, color=RUN_COLORS[run_label])

ax.set_xlabel("vocab_size")
ax.set_ylabel("test_cosine")
ax.set_title("test_cosine vs vocab_size, by architecture")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

*Notes: `char`'s single point per architecture isn't on this vocab_size axis - see the
dedicated char comparison below.*

## char: test_loss / test_cosine across architectures

In [ ]:
char_rows = [r for r in all_summary if r["tokenizer"] == "char"]
char_rows.sort(key=lambda r: list(RUNS.keys()).index(r["run_label"]))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = [r["run_label"] for r in char_rows]
colors = [RUN_COLORS[l] for l in labels]

axes[0].bar(labels, [r["test_loss"][0] for r in char_rows],
            yerr=[r["test_loss"][1] for r in char_rows], capsize=3, color=colors)
axes[0].set_ylabel("test_loss")
axes[0].set_title("char test_loss")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(labels, [r["test_cosine"][0] for r in char_rows],
            yerr=[r["test_cosine"][1] for r in char_rows], capsize=3, color=colors)
axes[1].set_ylabel("test_cosine")
axes[1].set_title("char test_cosine")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## num_parameters vs vocab_size, one line per architecture

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for run_label in RUNS:
    wp = sorted((r for r in all_summary if r["run_label"] == run_label and r["tokenizer"] == "wordpiece"),
                key=lambda r: r["vocab_size"])
    vocab_sizes = [r["vocab_size"] for r in wp]
    params = [r["num_parameters"] for r in wp]
    ax.plot(vocab_sizes, params, marker="o", label=run_label, color=RUN_COLORS[run_label])

ax.set_xlabel("vocab_size")
ax.set_ylabel("num_parameters")
ax.set_title("num_parameters vs vocab_size, by architecture")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Convergence (val_loss by epoch), split one panel per vocab_size

Split out on purpose - all 5 architectures x 15 configs in one chart would be unreadable.
Each panel below has (up to) 5 lines, one per architecture, for that single vocab_size.

In [ ]:
def history_filename(row):
    label = "char" if row["tokenizer"] == "char" else f"wordpiece_{int(row['vocab_size'])}"
    max_token_len = row["max_token_len"]
    filter_suffix = f"_n{int(max_token_len)}" if pd.notna(max_token_len) else ""
    pad_layout = row.get("pad_layout", "end")
    layout_suffix = f"_{pad_layout}" if pad_layout != "end" else ""
    hidden_dim = row.get("hidden_dim", DEFAULT_HIDDEN_DIM)
    num_conv_blocks = row.get("num_conv_blocks", DEFAULT_NUM_CONV_BLOCKS)
    arch_suffix = (
        f"_h{int(hidden_dim)}_d{int(num_conv_blocks)}"
        if (hidden_dim != DEFAULT_HIDDEN_DIM or num_conv_blocks != DEFAULT_NUM_CONV_BLOCKS)
        else ""
    )
    return f"{label}{filter_suffix}{layout_suffix}{arch_suffix}_seed{int(row['seed'])}.json"


records = []
missing = []
for _, row in raw.iterrows():
    history_path = run_dirs[row["run_label"]] / "history" / history_filename(row)
    if not history_path.exists():
        missing.append((row["run_label"], history_path.name))
        continue
    with open(history_path) as f:
        history = json.load(f)
    for entry in history:
        records.append({
            "run_label": row["run_label"],
            "tokenizer": row["tokenizer"],
            "vocab_size": row["vocab_size"],
            "epoch": entry["epoch"],
            "val_loss": entry["val"]["loss"],
        })

history_df = pd.DataFrame.from_records(records)
if missing:
    print(f"Missing {len(missing)} history file(s), e.g. {missing[:5]}")
print(f"Loaded {len(history_df)} (run, config, epoch) rows")

In [ ]:
panel_keys = (
    history_df[["tokenizer", "vocab_size"]]
    .drop_duplicates()
    .sort_values(["tokenizer", "vocab_size"], na_position="first")
    .to_records(index=False)
)

n = len(panel_keys)
ncols = min(4, n)
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False)

for ax, (tokenizer, vocab_size) in zip(axes.flat, panel_keys):
    subset = history_df[(history_df["tokenizer"] == tokenizer) & (history_df["vocab_size"] == vocab_size)]
    for run_label in RUNS:
        run_data = subset[subset["run_label"] == run_label].sort_values("epoch")
        if run_data.empty:
            continue
        ax.plot(run_data["epoch"], run_data["val_loss"], label=run_label, color=RUN_COLORS[run_label], linewidth=1.2)
    title = "char" if tokenizer == "char" else f"wordpiece {int(vocab_size)}"
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("epoch")
    ax.set_ylabel("val_loss")

for ax in axes.flat[n:]:
    ax.axis("off")

axes.flat[0].legend(fontsize=6)
plt.tight_layout()
plt.show()

*Notes: legend is only drawn once (top-left panel) to keep every other panel
uncluttered - the color mapping is the same across all panels (see the parameters
cell: `RUN_COLORS`).*